In [1]:
!pip install --upgrade keras

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.1 MB/s eta 0:00:00
  Attempting uninstall: keras
    Found existing installation: keras 2.15.0
    Uninstalling keras-2.15.0:
      Successfully uninstalled keras-2.15.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.15.0 requires keras<2.16,>=2.15.0, but you have keras 3.0.4 which is incompatible.


In [2]:
import keras

print(keras.__version__)

3.0.4


In [3]:
import numpy as np
from keras.datasets import mnist
from keras import Sequential
from keras.layers import Input, Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from keras.utils import to_categorical

# 指定亂數種子
np.random.seed(7)
# 載入資料集
(X_train, y_train), (X_test, y_test) = mnist.load_data()
# 建立2個資料集，一個數字小於 5，一個數字大於等於 5
X_train_lt5 = X_train[y_train < 5]
y_train_lt5 = y_train[y_train < 5]
X_test_lt5 = X_test[y_test < 5]
y_test_lt5 = y_test[y_test < 5]

X_train_gte5 = X_train[y_train >= 5]
y_train_gte5 = y_train[y_train >= 5] - 5
X_test_gte5 = X_test[y_test >= 5]
y_test_gte5 = y_test[y_test >= 5] - 5
# 將圖片轉換成 4D 張量
X_train_lt5 = X_train_lt5.reshape(
        (X_train_lt5.shape[0], 28, 28, 1)).astype("float32")
X_test_lt5 = X_test_lt5.reshape(
        (X_test_lt5.shape[0], 28, 28, 1)).astype("float32")
X_train_gte5 = X_train_gte5.reshape(
        (X_train_gte5.shape[0], 28, 28, 1)).astype("float32")
X_test_gte5 = X_test_gte5.reshape(
        (X_test_gte5.shape[0], 28, 28, 1)).astype("float32")
# 因為是固定範圍, 所以執行正規化, 從 0-255 至 0-1
X_train_lt5 = X_train_lt5 / 255
X_test_lt5 = X_test_lt5 / 255
X_train_gte5 = X_train_gte5 / 255
X_test_gte5 = X_test_gte5 / 255
# One-hot編碼
y_train_lt5 = to_categorical(y_train_lt5, 5)
y_test_lt5 = to_categorical(y_test_lt5, 5)
y_train_gte5 = to_categorical(y_train_gte5, 5)
y_test_gte5 = to_categorical(y_test_gte5, 5)
# 定義模型
model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(8, kernel_size=(3, 3),
                 activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(8, kernel_size=(3, 3), activation="relu"))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.25))
model.add(Dense(5, activation="softmax"))
model.summary()   # 顯示模型摘要資訊
print("--------------------------")
# 編譯模型
model.compile(loss="categorical_crossentropy", optimizer="adam",
              metrics=["accuracy"])
# 訓練模型
history = model.fit(X_train_lt5, y_train_lt5, validation_split=0.2,
                    epochs=5, batch_size=128, verbose=2)
print("--------------------------")
# 評估模型
loss, accuracy = model.evaluate(X_test_lt5, y_test_lt5, verbose=0)
print("測試資料集的準確度 = {:.2f}".format(accuracy))
print("--------------------------")
# 顯示各神經層
print(len(model.layers))
for i in range(len(model.layers)):
    print(i, model.layers[i])
# 凍結上層模型
for i in range(4):
    model.layers[i].trainable = False
print("--------------------------")
# 編譯模型
model.compile(loss="categorical_crossentropy", optimizer="adam",
              metrics=["accuracy"])
# 訓練模型
history = model.fit(X_train_gte5, y_train_gte5, validation_split=0.2,
                    epochs=5, batch_size=128, verbose=2)
print("--------------------------")
# 評估模型
loss, accuracy = model.evaluate(X_test_gte5, y_test_gte5, verbose=0)
print("測試資料集的準確度 = {:.2f}".format(accuracy))

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Layer (type)                       ┃ Output Shape                  ┃     Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                    │ (None, 26, 26, 8)             │          80 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ max_pooling2d (MaxPooling2D)       │ (None, 13, 13, 8)             │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ conv2d_1 (Conv2D)                  │ (None, 11, 11, 8)             │         584 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ max_pooling2d_1 (MaxPooling2D)     │ (None, 5, 5, 8)               │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ flatten (Flatten)                  │ (None, 200)                   │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dense (Dense)                      │ (None, 64)                    │      12,864 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dropout (Dropout)                  │ (None, 64)                    │           0 │
├────────────────────────────────────┼───────────────────────────────┼─────────────┤
│ dense_1 (Dense)                    │ (None, 5)                     │         325 │
└────────────────────────────────────┴───────────────────────────────┴─────────────┘

 Total params: 13,853 (54.11 KB)

 Trainable params: 13,853 (54.11 KB)

 Non-trainable params: 0 (0.00 B)

--------------------------
Epoch 1/5
192/192 - 11s - 55ms/step - accuracy: 0.8756 - loss: 0.4001 - val_accuracy: 0.9716 - val_loss: 0.0958
Epoch 2/5
192/192 - 2s - 10ms/step - accuracy: 0.9667 - loss: 0.1139 - val_accuracy: 0.9789 - val_loss: 0.0651
Epoch 3/5
192/192 - 1s - 4ms/step - accuracy: 0.9741 - loss: 0.0836 - val_accuracy: 0.9858 - val_loss: 0.0470
Epoch 4/5
192/192 - 1s - 3ms/step - accuracy: 0.9810 - loss: 0.0631 - val_accuracy: 0.9856 - val_loss: 0.0428
Epoch 5/5
192/192 - 1s - 3ms/step - accuracy: 0.9834 - loss: 0.0537 - val_accuracy: 0.9905 - val_loss: 0.0324
--------------------------
測試資料集的準確度 = 0.99
--------------------------
8
0 <Conv2D name=conv2d, built=True>
1 <MaxPooling2D name=max_pooling2d, built=True>
2 <Conv2D name=conv2d_1, built=True>
3 <MaxPooling2D name=max_pooling2d_1, built=True>
4 <Flatten name=flatten, built=True>
5 <Dense name=dense, built=True>
6 <Dropout name=dropout, built=True>
7 <Dense name=dense_1, built=True>
--------------------------
Epoch 1/